# Task 6 — Volleyball Ball Tracker 


In this notebook, we use **OpenCV** to track a volleyball in a match video.
We will:
- Detect the ball in each frame using colour detection
- Trace the ball's trajectory as it moves
- Count the number of players on each team


## Step 1 — Import Libraries


We first import all the tools we need:
- `cv2` → OpenCV, the main computer vision library
- `numpy` → for working with arrays (images are arrays)
- `matplotlib` → for displaying images inside Jupyter
- `deque` → a special list that auto-removes old items (used for ball trail)

## Define circularity function
we need a mathematical measure of how circular something is.

Formula:  circularity= 4π⋅area /perimeter^2
- ≈ 1 : perfect circle
- < 1 : irregular shape

Now , permeter should not be 0 - crash.

When can this happen?
- Very tiny contour
- Noise pixel
- Broken detection

## Define a function to count players 
A function that takes:
- frame:  original image (for drawing)
- mask : binary image (only team color visible)
- color : rectangle color (for drawing players)
  
Returns:
 number of detected players

1. A contour is the boundary (outline) of an object in an image
- cv.RETR_EXTERNAL : Contour retrieval mode- Only give me the outermost contours
- cv.CHAIN_APPROX_SIMPLE : Contour approximation method - Store only important boundary points not every pixel
- returns : contours, hierarchy

2. Initialize counter
3. Loop through contours
   - Area calculation : counts no of pixels 
   - Skip small blobs
   - Perimeter : Measures boundary length of contour
   - check p !=0
   - circularity check
   - Bounding Box
   - aspect ratio= width/ height  (Tall player< 1 , Square≈ 1, Wide objec > 1)
   - FINAL FILTER ( Not circular + Not too wide = likely a player)
   - cv.rectangle(frame, (x, y), (x + w, y + h), color, 2)
   - Increase count
4. return player count

how u computed the area threshold ?

print(area) -  Observe values - You’ll see something like: Noise(20–200) , Small blobs(200–400), Players(800–3000)

or use histogram based filtering

## GET VALID BALL
Takes all contours (from mask) & returns only valid ball-like contours

1. Create empty list - to store valid ball positions
2. Loop through contours
   - calculate area
   - then area filtering
   - perimeter
   - check p !=0
   - circularity
   - bounding box
   - aspect ratio ( for ball :  width ≈ height)
   - MAIN BALL CONDITION
   - enclosing circle
   - Moments - compute geometric position
   - Gives centroid (center of object)
   - Safety check - division by 0
   - Compute center
   - Store point


if area < 20 or area > 1500:

if circularity > 0.5

if 0.5 < aspect_ratio < 1.5

## Tracking the ball 
Inputs:
- valid_points → list of possible balls (from previous function)
- prev_center → last known ball position
- max_distance_threshold → how far ball is allowed to move
  
Goal: From multiple possible candidates, pick the correct ball based on 

1. no valid point case
2. If we have previous position
   - Sort points by distance (which points is closest to previous ball position)
   - Pick closest
   - Calculate distance again
   - Distance threshold check
     (Previous ball at (100,100) points : A - (110,110) :(real ball) , B - (500,500) (noise)
3. If NO previous position
   - Sort by area (Ball is usually biggest among small objects)
   - Return largest

~ Nearest Neighbor Tracking (basic tracking algorithm)


## Draw Trajectory 
Inputs:
- frame : current video frame (image)
- track_points : list (deque) of previous ball positions

1. Loop through points
    - Iterates from 2nd point to last
    - Handle missing  (Sometimes ball is not detected: track_points.appendleft(None) :[(100,100), (120,110), None, (150,130)])
    - draw line
        - frame : where to draw
        - track_points[i - 1]: start point
        - track_points[i] : end point
        - (0,255,255) : color (yellow)
        - 2 : thickness

## setup / initialization layer
1. Load video
2. if path wrong it wont load ( ret : false)
3. setup / initialization layer.
4. get frame width
5. get frame height
6. define codec - video encoding format : tells how vdo will be compressed
7. Create Video Writer (Creates output video file, You can write frames into it)
8. HSV Ranges ( team 1, team 2, ball)
9. tracking points storage : stores last seven ball position
10. Previous Center
11. max distance threshold - limits how far ball can move between frames

Why deque:  fast insert/remove & auto removes old points
 
Why maxlen=7: controls tail length


## MAIN LOOP
infinte loop - 
1. Read Frame
2. when video ends get out of loop
3. blur image - to remove noise , smooth
4. convert to hsv
5. mask for team 1
6. mask for team 2
7. mask for ball
8. Morphology Setup ( ellipse - natural )
9. erosion - removes noise
10. 10 . dilation - expands object , connect broken parts
11. final erosion
12. count players ( team 1 and 2)
13. display counters
14. ball contours
15. final ball points
16. track ball
17. if ball found ?
    - update prev center as this current best center
    - draw ball
    - same position
    -  if not found : track_points.appendleft(None)
18. draw trajectory
19. show frame
20. save frame
21. exit key



DOUBTS

1. ret ? A boolean (True/False) that tells whether a frame was successfully read.
2. writter ? An object used to save processed frames into a video file
3. upper and lower ?? HSV color ranges used to detect specific colors
4. morphology ?? Operations to clean and refine binary images (masks) 
   - erosion : removes white pixel from edges
   - dilation : adds white pixels(fill gaps , reconnects broken parts)
5. Most probable issues:
   - HSV ranges → wrong masks
   - Ball filtering too strict
   - Morphology removing ball
   - Area thresholds

In [43]:
import cv2 as cv
import numpy as np
import math
from collections import deque

def cal_circularity(area, perimeter):
    if perimeter == 0: 
        return 0
    else:
        return (4 * math.pi * area) / (perimeter * perimeter)

def count_players(frame, mask, color):
 
    contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    player_count = 0

    for contour in contours:
        area = cv.contourArea(contour)
        if area < 500: 
            continue
            
        perimeter = cv.arcLength(contour, True)
        if perimeter == 0:
            continue
            
        circularity = cal_circularity(area, perimeter)
        x, y, w, h = cv.boundingRect(contour)
        aspect_ratio = w / float(h)

        if circularity <= 0.65 and aspect_ratio < 1.5:
            cv.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            player_count += 1

    return player_count

def get_valid_ball_candidates(contours):
    candidates = []
    
    for c in contours:
        area = cv.contourArea(c)
        
        # print("\n--- NEW CONTOUR ---")
        #print("Area:", area)
        if area < 30 or area > 800:
            continue
            
        perimeter = cv.arcLength(c, True)
        if perimeter == 0:
            continue
            
        circularity = cal_circularity(area, perimeter)
        #print("Circularity:", circularity)
        x, y, w, h = cv.boundingRect(c)
        aspect_ratio = w / float(h)
        # print("Aspect Ratio:", aspect_ratio)

        if circularity > 0.65 and (0.7 < aspect_ratio < 1.3):
            # print("PASSED ALL CONDITIONS ")
            ((cx, cy), radius) = cv.minEnclosingCircle(c)
            M = cv.moments(c)
            if M["m00"] > 0:
                center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))
                candidates.append({
                    'contour': c,
                    'center': center,
                    'radius': int(radius),
                    'area': area
                })
        #else 
            #print("Rejected by SHAPE ")
    return candidates

def track_ball(valid_candidates, prev_center, max_distance_threshold):
    if len(valid_candidates) == 0:
        return None

    if prev_center is not None:
        valid_candidates.sort(key=lambda can: math.hypot(can['center'][0] - prev_center[0], can['center'][1] - prev_center[1]))
        closest_candidate = valid_candidates[0]
        
        dist = math.hypot(closest_candidate['center'][0] - prev_center[0], closest_candidate['center'][1] - prev_center[1])
        if dist <= max_distance_threshold:
            return closest_candidate
        else:
            return None 
    else:
        valid_candidates.sort(key=lambda can: can['area'], reverse=True)
        return valid_candidates[0]

def draw_tail(frame, track_points):
    for i in range(1, len(track_points)):
        if track_points[i - 1] is None or track_points[i] is None:
            continue
        cv.line(frame, track_points[i - 1], track_points[i], (0, 255, 255), 2)

video = cv.VideoCapture('Volleyball.mp4')
fps = int(video.get(cv.CAP_PROP_FPS))
width = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))

fourcc = cv.VideoWriter_fourcc(*'mp4v')
writer = cv.VideoWriter('output.mp4', fourcc, fps, (width, height))

lower_brazil = np.array([10, 100, 100]) 
upper_brazil = np.array([40, 255, 255]) 

lower_argentina = np.array([110, 50, 60]) 
upper_argentina = np.array([145, 130, 255]) 

lower_ball = np.array([15, 120, 120])
upper_ball = np.array([35, 255, 255])

track_points = deque(maxlen=7) 
prev_center = None
max_distance_threshold = 200 

while True:
    ret, frame = video.read()

    if not ret:
        print("Video processing complete.")
        break
    
    frame_blurred = cv.GaussianBlur(frame, (5, 5), 0)
    frame_hsv = cv.cvtColor(frame_blurred, cv.COLOR_BGR2HSV)

    mask_brazil = cv.inRange(frame_hsv, lower_brazil, upper_brazil)
    mask_argentina = cv.inRange(frame_hsv, lower_argentina, upper_argentina)
    mask_ball = cv.inRange(frame_hsv, lower_ball, upper_ball)

    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
    
    mask_brazil = cv.erode(mask_brazil, kernel, iterations=1)
    mask_ball = cv.erode(mask_ball, kernel, iterations=1)
    mask_argentina = cv.erode(mask_argentina, kernel, iterations=1)
    
    mask_brazil = cv.dilate(mask_brazil, kernel, iterations=3)
    mask_ball = cv.dilate(mask_ball, kernel, iterations=3)
    mask_argentina = cv.dilate(mask_argentina, kernel, iterations=3)
    
    mask_brazil = cv.erode(mask_brazil, kernel, iterations=1)
    mask_ball = cv.erode(mask_ball, kernel, iterations=1)
    mask_argentina = cv.erode(mask_argentina, kernel, iterations=1)

    brazil_player_count = count_players(frame, mask_brazil, (0, 255, 255))
    argentina_player_count = count_players(frame, mask_argentina, (255, 0, 0))

    count_text = f"BRAZIL: {brazil_player_count} ARGENTINA: {argentina_player_count}"
    cv.putText(frame, count_text, (10, 40), cv.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 0), 3)
    
    contours_ball, _ = cv.findContours(mask_ball, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    valid_candidates = get_valid_ball_candidates(contours_ball)
    
    best_candidate = track_ball(valid_candidates, prev_center, max_distance_threshold)

    if best_candidate is not None:
        prev_center = best_candidate['center']
        cv.circle(frame, best_candidate['center'], best_candidate['radius'], (0, 255, 0), 2)
        track_points.appendleft(prev_center)
    else:
        track_points.appendleft(None)

    draw_tail(frame, track_points)

    cv.imshow("Output", frame)
    writer.write(frame)
    
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
writer.release()
cv.destroyAllWindows()

Video processing complete.
